# Notebook 02 — Behavioral Compression Pipeline

This notebook covers Layer 2: entropy weighting, PCA-based dimensionality
reduction, gap-statistic cluster selection, and two-stage clustering.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from caps.acquisition.loaders import _generate_synthetic_data
from caps.acquisition.signal_matrix import SignalMatrix
from caps.compression.entropy_weighting import EntropyWeighter
from caps.compression.dimensionality import BehavioralCompressor
from caps.compression.clustering import TwoStageClustering
from caps.visualization.signal_plots import (
    plot_entropy_weights, plot_explained_variance, plot_silhouette
)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'serif'

In [ ]:
class _Cfg:
    observation_window = 90
    switching_threshold = 0.30
    variance_threshold = 0.72
    entropy_discretization_bins = 10
    max_clusters = 8
    min_silhouette = 0.45
    random_state = 42

cfg = _Cfg()
df = _generate_synthetic_data(n_consumers=500, random_state=42)
sm = SignalMatrix(cfg)
X, consumer_ids = sm.build(df)
print(f'X shape: {X.shape}')

## 2.1 — Entropy Weighting

In [ ]:
ew = EntropyWeighter(cfg)
weights, H = ew.fit_transform(X)
X_weighted = ew.apply(X, weights)

dim_names = [
    'Adoption Lag', 'Brand Switch', 'Price Elasticity',
    'Attr. Priority', 'Loyalty Retention', 'Category Spend', 'Trial Freq.'
][:X.shape[1]]

weight_df = pd.DataFrame({
    'Dimension': dim_names,
    'Entropy H_j': H.round(4),
    'Divergence d_j': ew.divergence_.round(4),
    'Weight w_j': weights.round(4),
})
print(weight_df.to_string(index=False))

plot_entropy_weights(weights=weights, entropy=H, feature_names=dim_names)

## 2.2 — PCA Dimensionality Reduction

In [ ]:
bc = BehavioralCompressor(cfg)
Z, components, evr, k = bc.fit_transform(X_weighted)

print(f'Components retained (k): {k}')
print(f'Cumulative explained variance: {evr[:k].sum():.4f}')
print()

loadings = bc.loadings_table(feature_names=dim_names)
loadings_df = pd.DataFrame(loadings).round(4)
print('PC Loadings:')
print(loadings_df)

plot_explained_variance(evr=evr, n_components=k, variance_threshold=0.72)

## 2.3 — Biplot: PC1 vs PC2 with Loadings

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
true_seg = df['true_segment'].values
seg_colors = {
    'High-Power Adopters': '#1a5276',
    'Flexible Mid-Tier': '#2e86c1',
    'Price-Opportunist': '#aed6f1',
    'Disengaged Low-Power': '#99a3a4',
}

for seg, color in seg_colors.items():
    mask = true_seg == seg
    ax.scatter(Z[mask, 0], Z[mask, 1], c=color, alpha=0.45, s=25,
               label=seg, edgecolors='white', linewidths=0.2)

scale = np.abs(Z).max() * 0.6
for i, name in enumerate(dim_names[:components.shape[1]]):
    ax.annotate(
        '', xy=(components[0, i]*scale, components[1, i]*scale), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5)
    )
    ax.text(components[0, i]*scale*1.12, components[1, i]*scale*1.12,
            name, fontsize=8, color='#c0392b', fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}% variance)', fontsize=10)
ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}% variance)', fontsize=10)
ax.set_title('Biplot — PC1 vs PC2 with Signal Loadings', fontsize=11, fontweight='bold')
ax.legend(fontsize=7, loc='best')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2.4 — Gap Statistic and Two-Stage Clustering

In [ ]:
cl = TwoStageClustering(cfg)
labels, centroids, sil = cl.fit(Z)

print(f'Optimal K: {cl.optimal_k_}')
print(f'Silhouette Score: {sil:.4f}')
print()
summary = cl.cluster_summary()
for k, info in summary.items():
    print(f'  Cluster {k}: n={info["size"]}, sil={info["mean_silhouette"]:.3f}')

In [ ]:
if cl.gap_statistics_ is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    k_range = np.arange(1, len(cl.gap_statistics_) + 1)
    ax.plot(k_range, cl.gap_statistics_, 'o-', color='#1a5276', linewidth=2, markersize=6)
    ax.axvline(cl.optimal_k_, color='#e74c3c', linestyle='--', linewidth=1.5,
               label=f'Selected K={cl.optimal_k_}')
    ax.set_xlabel('Number of Clusters K', fontsize=10)
    ax.set_ylabel('Gap Statistic', fontsize=10)
    ax.set_title('Gap Statistic — Optimal K Selection', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_silhouette(Z=Z, labels=labels, silhouette_samples=cl.silhouette_samples_)